# Task 3 — Fixed epoch-25 SAM screen

Run **folds 0 and 4 from scratch for exactly 25 epochs**, with MixUp **0.2** and SAM **rho 0.05**. Keep the original **30-epoch cosine learning-rate schedule**. Save and fully evaluate the epoch-25 model.

Use a **fresh Colab L4** matching 04ai. Push this notebook and its source files first. The original 04af parents, earlier comparisons and precision evidence must be on Drive. Run All stops after this one two-fold screen.

This stopping point was chosen after inspecting 04ai validation diagnostics. This is a follow-up check, not an independent blind test.

## 1. Colab GPU and repository


In [1]:
import os
import shutil
import subprocess
import sys
import time
import zipfile
from pathlib import Path

REPO_URL = "https://github.com/TrnLin/MLA2.git"
BRANCH = "fashion-analysis-and-cleanup"
CHECKOUT_DIR = Path("/content/MLA2")
DRIVE_MOUNT = Path("/content/drive")
DRIVE_PROJECT_DIR = DRIVE_MOUNT / "MyDrive/MLA2"
DATA_ZIP = DRIVE_PROJECT_DIR / "data/task3-data.zip"
LOCAL_DATA_ZIP = Path("/content/task3-data.zip")
DRIVE_TASK_DIR = DRIVE_PROJECT_DIR / "task3"
DRIVE_REGISTRY = DRIVE_TASK_DIR / "results/runs.csv"


def run_checked(command, *, cwd=None):
    command = [str(part) for part in command]
    print("$", " ".join(command), flush=True)
    return subprocess.run(command, cwd=cwd, check=True)


try:
    from google.colab import drive
except ImportError as exc:
    raise RuntimeError("Connect this notebook to a Google Colab GPU runtime first.") from exc

drive.mount(str(DRIVE_MOUNT), force_remount=False)
if (CHECKOUT_DIR / ".git").is_dir():
    remote_url = subprocess.check_output(
        ["git", "remote", "get-url", "origin"], cwd=CHECKOUT_DIR, text=True
    ).strip()
    if remote_url != REPO_URL:
        raise RuntimeError(f"{CHECKOUT_DIR} belongs to a different repository: {remote_url}")
    run_checked(["git", "fetch", "origin", BRANCH], cwd=CHECKOUT_DIR)
    run_checked(["git", "switch", BRANCH], cwd=CHECKOUT_DIR)
    run_checked(["git", "merge", "--ff-only", f"origin/{BRANCH}"], cwd=CHECKOUT_DIR)
elif CHECKOUT_DIR.exists():
    raise RuntimeError(f"{CHECKOUT_DIR} exists but is not a Git repository.")
else:
    run_checked(["git", "clone", "--branch", BRANCH, "--single-branch", REPO_URL, CHECKOUT_DIR])

commit = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=CHECKOUT_DIR, text=True).strip()
REPO_DIR = CHECKOUT_DIR / "core" if (CHECKOUT_DIR / "core/src/fashion").is_dir() else CHECKOUT_DIR
LOCAL_REGISTRY = REPO_DIR / "results/runs.csv"
print("Repository ready:", REPO_DIR)
print("Commit:", commit)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
$ git fetch origin task-3-gender-usage-classification
$ git switch task-3-gender-usage-classification
$ git merge --ff-only origin/task-3-gender-usage-classification
Repository ready: /content/MLA2
Commit: 5f0789506239944f2e836348b00bdc5e34d5658d


## 2. Teacher data and canonical split


In [2]:
def copy_teacher_zip_to_local_disk():
    if LOCAL_DATA_ZIP.is_file():
        try:
            with zipfile.ZipFile(LOCAL_DATA_ZIP) as existing:
                existing.infolist()
            return
        except zipfile.BadZipFile:
            LOCAL_DATA_ZIP.unlink()
    partial = LOCAL_DATA_ZIP.with_suffix(".zip.partial")
    for attempt in range(1, 4):
        partial.unlink(missing_ok=True)
        try:
            expected_bytes = DATA_ZIP.stat().st_size
            with DATA_ZIP.open("rb") as source, partial.open("wb") as target:
                shutil.copyfileobj(source, target, length=8 * 1024**2)
            if partial.stat().st_size != expected_bytes:
                raise OSError("The local ZIP copy is incomplete.")
            partial.replace(LOCAL_DATA_ZIP)
            return
        except OSError as error:
            partial.unlink(missing_ok=True)
            if attempt == 3:
                raise RuntimeError("Drive disconnected three times. Remount and retry.") from error
            drive.mount(str(DRIVE_MOUNT), force_remount=True)
            time.sleep(2)


copy_teacher_zip_to_local_disk()
teacher_dir = REPO_DIR / "data/raw/teacher"
required_files = (
    teacher_dir / "train/styles_train.csv",
    teacher_dir / "test/styles_prediction.csv",
)
image_dirs = (teacher_dir / "train/images_train", teacher_dir / "test/images_test")
image_suffixes = {".jpg", ".jpeg"}
with zipfile.ZipFile(LOCAL_DATA_ZIP) as archive:
    names = archive.namelist()
    if any(Path(name).is_absolute() or ".." in Path(name).parts for name in names):
        raise RuntimeError("The teacher archive contains an unsafe path.")
    expected_images = sum(
        name.startswith("data/raw/teacher/") and Path(name).suffix.lower() in image_suffixes
        for name in names
    )
    current_images = sum(
        path.suffix.lower() in image_suffixes for folder in image_dirs for path in folder.glob("*")
    )
    if current_images != expected_images or not all(path.is_file() for path in required_files):
        archive.extractall(REPO_DIR)

actual_images = sum(
    path.suffix.lower() in image_suffixes for folder in image_dirs for path in folder.glob("*")
)
if actual_images != expected_images or not all(path.is_file() for path in required_files):
    raise RuntimeError(f"Teacher data is incomplete: {actual_images:,}/{expected_images:,} images")

os.chdir(REPO_DIR)
os.environ["FASHION_PROJECT_ROOT"] = str(REPO_DIR)
if str(REPO_DIR / "src") not in sys.path:
    sys.path.insert(0, str(REPO_DIR / "src"))
(DRIVE_TASK_DIR / "results").mkdir(parents=True, exist_ok=True)
print(f"Teacher data ready: {actual_images:,} images")

Teacher data ready: 44,441 images


## 3. Freeze the 25-epoch budget and check the parents

In 04ai, the saved epoch-25 diagnostics showed pooled validation F1 **79.86%**, mean clean gap **9.94 points**, and **361/707 Unisex items** found correctly. The final epoch-30 model had a larger gap and found five fewer Unisex items. Those earlier scores had no saved checkpoint and lacked full IEEE and corruption checks.

This trial trains **25 epochs** and saves the **final epoch-25 checkpoint**. It is a fixed budget, not validation-based early stopping. Keep `CosineAnnealingLR(T_max=30)` and the same seed so the first 25 learning-rate values match 04ai. Do not compress the cosine schedule to 25 epochs or warm-start from the epoch-30 checkpoint. The target minimum rate remains **0.00001**, but the last training epoch does not yet reach that rate.

Keep SAM's two gradient passes on the **same augmented mixed batch and dropout mask**, first-pass BatchNorm running buffers only, and one AdamW update per batch. Restore exact original weights before the second-gradient update. Non-finite losses or gradients fail the run.

Keep corrected gender labels, canonical folds, full images, widths `[32, 64, 128, 256]`, **390,181 parameters**, GeM p=3, dropout **0.30**, grayscale **0.10**, translation ±2 px at probability 0.50, mild darkening at probability 0.25, batch **128**, seed **2753**, learning rate **0.001**, weight decay **0.0001**, MixUp **0.2**, and no class or sample weights. Parents supply comparison evidence only.

Save clean training and validation class scores at **epochs 10, 15, 20 and 25**. Online mixed-input F1 stays blank. Full final comparisons use matching name-truth labels and IEEE FP32, including all corruption checks and the original teacher-label diagnostic.

The direct scoring baseline stays **04af MixUp 0.2**. All 19 checks are unchanged. The completed 04ai screen remains failed; this new run has its own name, configuration hash, registry rows and output folder. The new audit binds the 25-epoch budget, 30-epoch scheduler and verification code.

In [3]:
from fashion.data.gender_name_truth import build_gender_name_truth_variant
from fashion.train.task3_gender_sam25 import (
    check_gender_sam25_sources,
    run_gender_sam25_screen,
)

G2_DIR = DRIVE_TASK_DIR / "experiments/t3_gender_v2_g2_translation/gender"
E6_DIR = DRIVE_TASK_DIR / "experiments/t3_gender_e6_gem_p3/gender"
DROPOUT_DIR = DRIVE_TASK_DIR / "experiments/t3_gender_dropout_030/gender"
DARKENING_DIR = DRIVE_TASK_DIR / "experiments/t3_gender_dropout_030_mild_darkening/gender"
GRAYSCALE_DIR = (
    DRIVE_TASK_DIR / "experiments/t3_gender_dropout_030_mild_darkening_grayscale_010/gender"
)
NAME_TRUTH_DIR = (
    DRIVE_TASK_DIR / "experiments/t3_gender_name_truth_dropout_030_grayscale_010/gender"
)
MIXUP_DIR = DRIVE_TASK_DIR / "experiments/t3_gender_name_truth_mixup_alpha020/gender"
PRECISION_DIR = DRIVE_TASK_DIR / "diagnostics/gender_precision/20260905T085822668071Z"

summary = build_gender_name_truth_variant(REPO_DIR)
print("Changed gender labels:", summary["changed_labels"])
print("Unclear names kept:", summary["no_cue_rows"] + summary["multiple_cue_rows"])
print("Fold label changes:", summary["folds"])
sources, classes, spec, evidence = check_gender_sam25_sources(
    g2_directory=G2_DIR,
    e6_directory=E6_DIR,
    dropout_directory=DROPOUT_DIR,
    darkening_directory=DARKENING_DIR,
    grayscale_directory=GRAYSCALE_DIR,
    name_truth_directory=NAME_TRUTH_DIR,
    mixup_directory=MIXUP_DIR,
    source_registry_path=DRIVE_REGISTRY,
    precision_directory=PRECISION_DIR,
    root=REPO_DIR,
)
assert spec.classifier_dropout == 0.30
assert spec.to_dict()["grayscale_probability"] == 0.10
print("Verified source runs:", {name: len(runs) for name, runs in sources.items()})
print(
    "Direct alpha 0.2 parents:",
    {fold: run["run_id"] for fold, run in sources["MixUp20"].items()},
)
print("Frozen recipe:", spec.to_dict())
print("Output:", DRIVE_TASK_DIR / spec.artifact_dir / "gender")
print("MixUp policy:", spec.to_dict()["mixup_policy"])
print("Additional improvement rules:", spec.to_dict()["improvement_rules"])

print("SAM policy:", spec.to_dict()["sam_policy"])
print("Training epochs:", spec.to_dict()["training_epochs"])
print("Cosine schedule T_max:", spec.to_dict()["cosine_t_max"])

Changed gender labels: 350
Unclear names kept: 1320
Fold label changes: [{'fold': 0, 'validation_rows': 6553, 'changed_validation_labels': 73, 'changed_training_labels': 277}, {'fold': 1, 'validation_rows': 6556, 'changed_validation_labels': 85, 'changed_training_labels': 265}, {'fold': 2, 'validation_rows': 6553, 'changed_validation_labels': 58, 'changed_training_labels': 292}, {'fold': 3, 'validation_rows': 6554, 'changed_validation_labels': 50, 'changed_training_labels': 300}, {'fold': 4, 'validation_rows': 6557, 'changed_validation_labels': 84, 'changed_training_labels': 266}]
Verified source runs: {'G2': 5, 'E6': 5, 'Drop30': 2, 'Drop30Dark': 2, 'Gray10': 2, 'NameTruth': 2, 'MixUp20': 2}
Direct alpha 0.2 parents: {0: 't3_gender_name_truth_mixup_alpha020_gender_smallcnngem3_f0_s2753_4566a61e0e2d_20260906T070013Z8f7ca5', 4: 't3_gender_name_truth_mixup_alpha020_gender_smallcnngem3_f4_s2753_4566a61e0e2d_20260906T071118Z37920b'}
Frozen recipe: {'name': 'gender_name_truth_mixup_alpha020

## 4. Preserve the old guards and require further improvement

Keep the **14 non-F1 G2/E6 checks**, with the same labels and IEEE evaluation:

- Relative pooled, fold and class F1 changes and the paired whole-family bootstrap interval are diagnostic only. Keep 10,000 draws within folds, seed 2753; save the five replaced G2 F1 checks separately as `diagnostic_checks`.
- The mean clean gap must fall by at least 0.050 versus G2, and both folds' gaps must shrink.
- NLL may rise by at most 0.020 and ECE by at most 0.010 versus G2.
- Keep corruption guards versus matched E6: translation-induced F1 change improves by at least 0.030; each other standard corruption worsens by at most 0.020.
- Exactly 390,181 parameters and peak allocated GPU memory strictly below 3,000,000,000 bytes. Report training time and latency. A memory failure stops before another fold begins.

Add **5 checks**, using completed 04af alpha 0.2 for the gap and recall comparisons:

- Mean clean gap must shrink by **at least 0.020**: approximately **0.129133 → at most 0.109133**.
- Each fold's clean gap must shrink.
- Pooled validation macro-F1 must be **at least 0.74 (74%)**. A drop from the parent's 80.89% is allowed.
- **Unisex recall must not fall** versus 04af (approximately **0.510608**).

Use exact freshly matched parent scores, not these rounded numbers. **All 19 checks must pass.** These targets are chosen before this new fit; they do not rewrite the result of 04af. Read confidence quality and raw corruption scores as well. The 74% floor allows a validation F1 drop in exchange for a smaller clean gap.


In [4]:
result = run_gender_sam25_screen(
    g2_directory=G2_DIR,
    e6_directory=E6_DIR,
    dropout_directory=DROPOUT_DIR,
    darkening_directory=DARKENING_DIR,
    grayscale_directory=GRAYSCALE_DIR,
    name_truth_directory=NAME_TRUTH_DIR,
    mixup_directory=MIXUP_DIR,
    source_registry_path=DRIVE_REGISTRY,
    precision_directory=PRECISION_DIR,
    output_root=DRIVE_TASK_DIR,
    registry_path=DRIVE_REGISTRY,
    registry_mirrors=(LOCAL_REGISTRY,),
    root=REPO_DIR,
)
print("Screen:", result["status"])
print("Label basis:", result.get("comparison_label_basis"))
for row in result.get("folds", []):
    print(
        "Fold",
        row["fold"],
        "train F1:",
        row["candidate_train_f1"],
        "validation F1:",
        row["candidate_validation_f1"],
        "gap:",
        row["candidate_gap"],
    )
for gate in result.get("checks", []):
    if gate["status"] != "pass":
        print(gate)
if "reason" in result:
    print(result["reason"])
if "incremental_comparison" in result:
    incremental = result["incremental_comparison"]
    print("Direct alpha 0.2 parents:", result["direct_parent_run_ids"])
    print("F1 change versus alpha 0.2 on the same labels:", incremental["validation_delta"])
    print("Paired 95% interval:", incremental["validation_interval"])
    print("Class F1 changes:", incremental["class_f1_delta"])
    print("Induced corruption changes:", incremental["mean_induced_change_delta"])
if "incremental_comparison" in result:
    for row in result["incremental_comparison"]["folds"]:
        print("Direct-parent gap change, fold", row["fold"], ":", row["delta_gap"])

IEEE evaluation: t3_gender_v2_g2_translation_gender_smallcnngem3_f0_s2753_cb072542dbdc_20260904T135102Z6698e6
IEEE evaluation: t3_gender_v2_g2_translation_gender_smallcnngem3_f4_s2753_cb072542dbdc_20260904T140011Z2211b1
IEEE evaluation: t3_gender_e6_gem_p3_gender_smallcnngem3_f0_s2753_a8c09286451b_20260831T090059Z0bab1f
IEEE evaluation: t3_gender_e6_gem_p3_gender_smallcnngem3_f4_s2753_a8c09286451b_20260831T093553Z63b5fd
IEEE evaluation: t3_gender_dropout_030_mild_darkening_grayscale_010_gender_smallcnngem3_f0_s2753_eb37119b7e68_20260905T144455Ze47ef6
IEEE evaluation: t3_gender_dropout_030_mild_darkening_grayscale_010_gender_smallcnngem3_f4_s2753_eb37119b7e68_20260905T145521Z7d0a34
IEEE evaluation: t3_gender_name_truth_mixup_alpha020_gender_smallcnngem3_f0_s2753_4566a61e0e2d_20260906T070013Z8f7ca5
IEEE evaluation: t3_gender_name_truth_mixup_alpha020_gender_smallcnngem3_f4_s2753_4566a61e0e2d_20260906T071118Z37920b
[task3] preparing target=gender fold=0: train=26,220 (before selection=26,

## 5. Stop and review

Stop after the two 25-epoch runs. Keep 04af as the current candidate unless **all 19 checks pass**. A score below **74%** or a drop in Unisex recall fails this screen. Do not automatically try another stopping epoch, more folds, refits or held-out evaluation.

Output: `MyDrive/MLA2/task3/experiments/t3_gender_name_truth_mixup_alpha020_sam005_epoch25/gender`.

- Every fit enters the shared registry before its first optimizer step. `final_epoch.pt` must record **selected_epoch = epochs_completed = 25**, with `early_stopped = false` in metrics.
- `history.csv` must contain exactly **25 epochs** and the first 25 rates from the **30-epoch cosine schedule**. The run cannot reuse a 30-epoch receipt or checkpoint.
- `sam_training.json` and `mixup_training.json` must agree on row and batch coverage. SAM uses two gradient passes and one optimizer update per batch.
- `clean_epoch_diagnostics.json` stores full class scores at 10, 15, 20 and 25. Separate IEEE evaluations and corruption scores determine the final decision.
- Read `screen_decision.json`, `incremental_comparison.json` and `clean_gap_comparison.csv`. The inherited `dropout_*` comparison fields refer to the 04af alpha 0.2 parents.
- `source_audit.json`, `label_variant/`, saved predictions, metrics, hashes and the teacher-label diagnostic preserve the evidence.

The promising epoch-25 scores motivated this trial. They do not prove that this new screen will pass.